# Story 11: Adaptive Workflows - the Next Step Depends on the Results
Your workflow is not known in advance: a convergence test which runs until the result stops changing, a curve which is
refined where it varies most, or a screening where follow-up calculations start for whatever finished first. You want
to write the decision logic in plain Python - `while` loops, `if` statements, recursion - and still run the
calculations in parallel on the cluster.

**With plain SLURM** a batch script cannot look at results it has not computed yet. Adaptive workflows end up as a
driver script which submits a batch of jobs, polls `squeue` until they finish, parses the outputs, decides and submits
the next batch - with the cluster idling between the rounds. Static workflow graphs (job arrays, `--dependency`)
cannot express decisions at all.

**With executorlib** the futures returned by `submit()` are ordinary Python objects. Your driver logic simply waits
for the results it needs to make the next decision, while everything independent keeps running. `asyncio` turns
futures into awaitables for recursive refinement, and `batched()` groups results by completion order.

This notebook has to run **inside a SLURM allocation** with at least two cores, for example started with
`salloc -n 2`. With a `SlurmClusterExecutor` the same patterns work from the login node, with one SLURM job per call.

## Cluster-specific settings
**Adapt the following cell to your cluster and allocation.** The values below are the ones used to test this notebook in a two-core allocation of a single-node SLURM test cluster with MPICH. For OpenMPI >= 5 use `["--mpi=pmix"]`; set `max_cores` to the number of cores of your allocation.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster and allocation.
slurm_cmd_args = ["--mpi=pmi2"]  # additional arguments for srun, e.g. the MPI plugin
max_cores = 2  # number of CPU cores of the allocation executorlib may use

## Pattern 1: A convergence loop
The plane-wave cutoff of a DFT calculation is increased until the energy changes by less than a tolerance. Each round
compares two cutoffs which are computed in parallel; the futures are kept in a dictionary, so no cutoff is computed
twice.

In [ ]:
import time
from executorlib import SlurmJobExecutor


def compute_energy(cutoff):
    # Stand-in for a DFT calculation whose total energy converges with the plane-wave cutoff.
    time.sleep(0.5)
    return -10.0 + 50.0 / cutoff**2


tolerance = 1e-4
future_dict = {}

with SlurmJobExecutor(max_cores=max_cores, resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args}) as exe:

    def energy(cutoff):
        if cutoff not in future_dict:
            future_dict[cutoff] = exe.submit(compute_energy, cutoff)
        return future_dict[cutoff]

    cutoff = 200
    while True:
        e_low, e_high = energy(cutoff), energy(cutoff + 100)  # both run in parallel
        difference = abs(e_high.result() - e_low.result())
        print(f"cutoff {cutoff}: energy difference {difference:.2e}")
        if difference < tolerance:
            break
        cutoff += 100

print("converged cutoff:", cutoff)

## Pattern 2: Recursive refinement with `asyncio`
An energy curve is sampled adaptively: an interval is bisected only where the energy at the midpoint deviates from
the linear interpolation, so the calculations are spent where the curve is curved. `asyncio.wrap_future()` turns an
executorlib future into an awaitable, and `asyncio.gather()` refines both halves of an interval at the same time - so
independent branches of the recursion run in parallel on the cluster.

In [ ]:
import asyncio


def potential_energy(distance):
    # Stand-in for a calculation of the energy of a dimer at a given distance.
    time.sleep(0.3)
    return 4 * 0.0104 * ((3.4 / distance) ** 12 - (3.4 / distance) ** 6)


async def evaluate(exe, distance):
    return await asyncio.wrap_future(exe.submit(potential_energy, distance))


async def refine(exe, x_left, e_left, x_right, e_right, depth, tolerance=5e-4):
    x_mid = (x_left + x_right) / 2
    e_mid = await evaluate(exe, x_mid)
    if depth == 0 or abs(e_mid - (e_left + e_right) / 2) < tolerance:
        return [(x_mid, e_mid)]
    left, right = await asyncio.gather(
        refine(exe, x_left, e_left, x_mid, e_mid, depth - 1, tolerance),
        refine(exe, x_mid, e_mid, x_right, e_right, depth - 1, tolerance),
    )
    return left + [(x_mid, e_mid)] + right

Jupyter supports `await` directly in a cell; in a Python script wrap the call in `asyncio.run()`.

In [ ]:
with SlurmJobExecutor(max_cores=max_cores, resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args}) as exe:
    x_min, x_max = 3.4, 6.0
    e_min, e_max = await asyncio.gather(evaluate(exe, x_min), evaluate(exe, x_max))
    curve = [(x_min, e_min)] + await refine(exe, x_min, e_min, x_max, e_max, depth=3) + [(x_max, e_max)]

print(f"{len(curve)} points:")
for distance, energy_value in curve:
    print(f"{distance:.3f} {energy_value:+.5f}")

## Pattern 3: Start follow-up work in the order results arrive
In a screening, the run time of the individual candidates varies. Rather than waiting for all of them, `batched()`
groups the futures into batches of `n` in the order in which they finish. Each batch is passed to a follow-up
calculation which starts as soon as its batch is complete, while slower candidates are still running.

In [ ]:
def screen(candidate):
    # Stand-in for a cheap screening calculation with a candidate-dependent run time.
    time.sleep(0.2 * (candidate % 3))
    return {"candidate": candidate, "score": (candidate * 37) % 11}


def refine_best(batch):
    # Stand-in for an expensive follow-up calculation for the most promising candidate of a batch.
    best = min(batch, key=lambda result: result["score"])
    time.sleep(0.5)
    return best["candidate"]


with SlurmJobExecutor(max_cores=max_cores, resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args}) as exe:
    screening_lst = [exe.submit(screen, candidate) for candidate in range(9)]
    refined_lst = [exe.submit(refine_best, batch) for batch in exe.batched(screening_lst, n=3)]
    print("refined candidates:", [refined.result() for refined in refined_lst])

## What you learned
* Futures can drive plain Python control flow - loops and `if` statements decide what to submit next.
* `asyncio.wrap_future()` and `asyncio.gather()` turn recursive algorithms into parallel workflows.
* `batched()` groups futures by completion order, so follow-up steps start without waiting for the slowest task.

**Reference:** [advanced scheduling](https://executorlib.readthedocs.io/en/latest/1-single-node.html#advanced-scheduling) including
[runtime-dependent batching](https://executorlib.readthedocs.io/en/latest/1-single-node.html#runtime-dependent-batching) and
[recursion](https://executorlib.readthedocs.io/en/latest/1-single-node.html#recursion), and the [HPC Job Executor](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html).